In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator
from data_preprocessing import create_train_test_val_sets, read_processed_data


In [2]:
#Create test train splits
x_mendeley, y_mendeley = read_processed_data(r"..\data\processed\mendeley_processed.csv")
x_phiusiil, y_phiusiil= read_processed_data(r"..\data\processed\phiusiil_processed.csv")

mendeley_sets = create_train_test_val_sets(x_mendeley,y_mendeley, label_col="Label", test_size=0.2, n_splits=5)
phiusiil_sets = create_train_test_val_sets(x_phiusiil,y_phiusiil, label_col="Label", test_size=0.2, n_splits=5)

KeyboardInterrupt: 

### Tuning Classifiers

In [ ]:
#XGBoost
def optimize_xgboost(X: pd.DataFrame, y: pd.Series, dataset: str) -> RandomizedSearchCV:
    scale_weights = [1.0]
    counts = y.value_counts(normalize=True)
    if dataset == 'mendeley':
        scale_weights.append(counts[1]/counts[0])
    elif dataset == 'phiusiil':
        scale_weights.append(counts[0]/counts[1])
    
    params = {
        'max_depth': [4, 5, 6, 8, 10],
        'min_child_weight': [1, 3, 5, 7],
        'gamma': [0, 0.1, 0.2, 0.4],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
        'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.4],
        'n_estimators': [100, 300, 500, 750, 1000],
        'scale_pos_weight': scale_weights
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    xgb = XGBClassifier(random_state=42)
    random_search = RandomizedSearchCV(xgb, param_distributions=params, random_state=42, cv=skf.split(X, y))
    random_search.fit(X, y)

    print('\n Best hyperparameters:')
    print(random_search.best_params_)

    return random_search

print("Running hyperparameter tuning using Mendeley Dataset:")
xgboost_mendeley = optimize_xgboost(mendeley_sets["x_train_val"], mendeley_sets["y_train_val"], 'mendeley')
print("Running hyperparameter tuning using Phiusiil Dataset:")
xgboost_phiusiil = optimize_xgboost(phiusiil_sets["x_train_val"], phiusiil_sets["y_train_val"], 'phiusiil')


In [ ]:
from sklearn.metrics import mean_absolute_error, classification_report
def train_no_feature_selection(dataset, model):
    stratified_scores = []
    all_y_val = []
    all_y_pred = []
    #Training and Testing for both xgb classifiers
    for train_idx, val_idx in dataset["cv_splits"]:
        x_train, x_val = dataset["x_train_val"].iloc[train_idx], dataset["x_train_val"].iloc[val_idx]
        y_train, y_val = dataset["y_train_val"].iloc[train_idx], dataset["y_train_val"].iloc[val_idx]
        
        model.fit(x_train, y_train)
        y_pred = model.predict(x_val)
        stratified_scores.append(mean_absolute_error(y_val, y_pred))

        all_y_val.extend(y_val)
        all_y_pred.extend(y_pred)

    print(f"Mean MAE: {np.mean(stratified_scores):.4f}, Std MAE: {np.std(stratified_scores):.4f}")
    print('Confusion Matrix:')
    print(classification_report(all_y_val, all_y_pred))
    

print('Mendeley Results:')
train_no_feature_selection(mendeley_sets, xgboost_mendeley.best_estimator_)

print('Phiusiil Results:')
train_no_feature_selection(phiusiil_sets, xgboost_phiusiil.best_estimator_)



In [3]:
import pandas as pd
import numpy as np
from collections import Counter

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score


# ==================================================
# 1. Load raw datasets
# ==================================================
mendeley_df = pd.read_csv(r"..\data\raw\Mendeley Dataset.csv")
phiusiil_df = pd.read_csv(r"..\data\raw\PhiUSIIL Dataset.csv")


# ==================================================
# 2. Prepare datasets
#    - Mendeley: keep numeric features only
#    - PhiUSIIL: drop raw identity/text fields
# ==================================================
def prepare_mendeley(df: pd.DataFrame):
    """
    Use numeric phishing features only.
    Assumes target column is named 'Label'.
    """
    if "Label" not in df.columns:
        raise ValueError("Mendeley dataset must contain a 'Label' column.")

    y = df["Label"].copy()
    X = df.drop(columns=["Label"]).select_dtypes(include=[np.number]).fillna(0)

    return X, y


def prepare_phiusiil(df: pd.DataFrame):
    """
    Use only the structured numeric phishing features.
    Drop raw identity/text columns to avoid memorization:
    FILENAME, URL, Domain, TLD, Title
    """
    if "Label" not in df.columns:
        raise ValueError("PhiUSIIL dataset must contain a 'Label' column.")

    cols_to_drop = ["FILENAME", "URL", "Domain", "TLD", "Title", "Label"]
    existing_drop_cols = [col for col in cols_to_drop if col in df.columns]

    y = df["Label"].copy()
    X = df.drop(columns=existing_drop_cols).select_dtypes(include=[np.number]).fillna(0)

    return X, y


x_mendeley, y_mendeley = prepare_mendeley(mendeley_df)
x_phiusiil, y_phiusiil = prepare_phiusiil(phiusiil_df)

print("Mendeley shape:", x_mendeley.shape)
print("Phiusiil shape:", x_phiusiil.shape)
print("Label in Mendeley features?", "Label" in x_mendeley.columns)
print("Label in Phiusiil features?", "Label" in x_phiusiil.columns)


# ==================================================
# 3. Create train / validation / test splits
# ==================================================
def create_train_test_val_sets(x, y, test_size=0.2, n_splits=3, random_state=42):
    x_train_val, x_test, y_train_val, y_test = train_test_split(
        x, y, test_size=test_size, stratify=y, random_state=random_state
    )

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    cv_splits = [(train_idx, val_idx) for train_idx, val_idx in skf.split(x_train_val, y_train_val)]

    print(
        f"Train/validation/test split prepared: "
        f"{len(y_train_val)} instances for training and validation, "
        f"{len(y_test)} instances for testing"
    )
    print(f"Stratified {n_splits}-fold CV splits created.")

    return {
        "x_train_val": x_train_val.reset_index(drop=True),
        "y_train_val": y_train_val.reset_index(drop=True),
        "x_test": x_test.reset_index(drop=True),
        "y_test": y_test.reset_index(drop=True),
        "cv_splits": cv_splits
    }


mendeley_sets = create_train_test_val_sets(x_mendeley, y_mendeley, test_size=0.2, n_splits=3)
phiusiil_sets = create_train_test_val_sets(x_phiusiil, y_phiusiil, test_size=0.2, n_splits=3)


# ==================================================
# 4. Build Logistic Regression pipeline
# ==================================================
def build_logistic_pipeline():
    """
    MaxAbsScaler is safe for feature ranges like these and lighter than
    standard normalization for this use case.
    saga handles larger / higher-dimensional problems better than lbfgs.
    """
    return Pipeline([
        ("scaler", MaxAbsScaler()),
        ("model", LogisticRegression(
            solver="saga",
            class_weight="balanced",
            max_iter=5000,
            tol=1e-3,
            random_state=42
        ))
    ])


# ==================================================
# 5. Hyperparameter tuning
# ==================================================
def optimize_logistic_regression(X, y):
    """
    Tune Logistic Regression using inner CV.
    """
    params = {
        "model__C": [0.01, 0.1, 1, 10]
    }

    inner_cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

    grid_search = GridSearchCV(
        estimator=build_logistic_pipeline(),
        param_grid=params,
        cv=inner_cv,
        scoring="f1",
        n_jobs=1,          # helps reproducibility
        error_score="raise"
    )

    grid_search.fit(X, y)

    print("Best hyperparameters:")
    print(grid_search.best_params_)

    return grid_search


# ==================================================
# 6. Nested CV evaluation + final hold-out test
# ==================================================
def train_no_feature_selection(dataset, dataset_name):
    """
    Outer CV:
        evaluate model
    Inner CV:
        tune hyperparameters

    Then choose the most common best C and evaluate on the hold-out test set.
    """
    fold_scores = []
    all_y_val = []
    all_y_pred = []
    best_cs = []

    for fold_num, (train_idx, val_idx) in enumerate(dataset["cv_splits"], start=1):
        x_train = dataset["x_train_val"].iloc[train_idx]
        x_val = dataset["x_train_val"].iloc[val_idx]
        y_train = dataset["y_train_val"].iloc[train_idx]
        y_val = dataset["y_train_val"].iloc[val_idx]

        print(f"\n{dataset_name} Fold {fold_num}: tuning hyperparameters...")
        tuned_model = optimize_logistic_regression(x_train, y_train)

        best_c = tuned_model.best_params_["model__C"]
        best_cs.append(best_c)
        print(f"{dataset_name} Fold {fold_num} best C: {best_c}")

        y_pred = tuned_model.best_estimator_.predict(x_val)

        fold_scores.append(f1_score(y_val, y_pred))
        all_y_val.extend(y_val)
        all_y_pred.extend(y_pred)

    print(f"\n{dataset_name} Nested CV Results")
    print(f"Mean F1: {np.mean(fold_scores):.4f}, Std F1: {np.std(fold_scores):.4f}")
    print("Validation Results:")
    print(classification_report(all_y_val, all_y_pred))

    # Pick the most common best C across folds
    final_c = Counter(best_cs).most_common(1)[0][0]
    print(f"{dataset_name} Final chosen C for hold-out test: {final_c}")

    final_model = Pipeline([
        ("scaler", MaxAbsScaler()),
        ("model", LogisticRegression(
            C=final_c,
            solver="saga",
            class_weight="balanced",
            max_iter=5000,
            tol=1e-3,
            random_state=42
        ))
    ])

    final_model.fit(dataset["x_train_val"], dataset["y_train_val"])
    y_test_pred = final_model.predict(dataset["x_test"])

    print(f"\n{dataset_name} Test Set Results:")
    print(classification_report(dataset["y_test"], y_test_pred))


# ==================================================
# 7. Run experiments
# ==================================================
print("\n========== Mendeley ==========")
train_no_feature_selection(mendeley_sets, "Mendeley")

print("\n========== Phiusiil ==========")
train_no_feature_selection(phiusiil_sets, "Phiusiil")


Mendeley shape: (247950, 41)
Phiusiil shape: (235795, 50)
Label in Mendeley features? False
Label in Phiusiil features? False
Train/validation/test split prepared: 198360 instances for training and validation, 49590 instances for testing
Stratified 3-fold CV splits created.
Train/validation/test split prepared: 188636 instances for training and validation, 47159 instances for testing
Stratified 3-fold CV splits created.

========== Mendeley ==========

Mendeley Fold 1: tuning hyperparameters...
Best hyperparameters:
{'model__C': 10}
Mendeley Fold 1 best C: 10

Mendeley Fold 2: tuning hyperparameters...
Best hyperparameters:
{'model__C': 10}
Mendeley Fold 2 best C: 10

Mendeley Fold 3: tuning hyperparameters...
Best hyperparameters:
{'model__C': 10}
Mendeley Fold 3 best C: 10

Mendeley Nested CV Results
Mean F1: 0.7857, Std F1: 0.0002
Validation Results:
              precision    recall  f1-score   support

           0       0.78      0.86      0.82    102833
           1       0.83  